# Check and relabel masks

In [ ]:
import napari, tifffile as tiff

In [ ]:
L = tiff.imread('/Volumes/mshahbazi-group/rsakata/EXP50/cellpose_withstruc/mask/EXP50_DAPI_Phalloindin_GFP_mcherry_250825_2_serie_15 - 8-1_mask.tif')   # 2D: YX, 3D: ZYX
v = napari.Viewer(ndisplay=3 if L.ndim==3 else 2)
v.add_labels(L, name="labels")

In [ ]:
# --------- Imports ---------
import os, glob, natsort, math, json, time
import numpy as np
import pandas as pd
import skimage.io as skio
from skimage.measure import label, regionprops
from cellpose import models, core, io

import matplotlib
matplotlib.use("Agg")           # headless backend
import matplotlib.pyplot as plt
plt.ioff()

In [ ]:
img = skio.imread('/Volumes/mshahbazi-group/rsakata/EXP56/LIF2TIF/setA/cropped/EXP56_setA_DAPI_GATA3_GFP_ZO1_NANAOG_serie_12 - 2_4_multi-1.tif')
shape = img.shape

print(f"{shape}" )

In [ ]:
(113, 5, 512, 512)

## 1. Extract summary files

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)


In [ ]:
root_dir <- "/Volumes/mshahbazi-group/rsakata/EXP50/cellpose_withstruc/cellpose_intensities"

# Find matching CSVs in all subfolders
csv_paths <- list.files(
  path = root_dir,
  pattern = "\\.csv$",
  recursive = TRUE,
  full.names = TRUE
)

if (length(csv_paths) == 0) stop("No matching files found.")

# Read and bind
combined_df <- purrr::map_dfr(csv_paths, ~ readr::read_csv(.x, show_col_types = FALSE))


combined_df <- combined_df %>%
  rename(image = sample) %>%
  mutate(sample = str_replace(image, "[_-].*$", ""))
combined_df$sample <- as.character(combined_df$sample)


In [ ]:
#merge with sample sheet
sample_sheet <- read_csv("/Volumes/mshahbazi-group/rsakata/EXP50/cellpose_withstruc/code/sample_sheet.csv", show_col_types = FALSE)

sample_sheet$sample <- as.character(sample_sheet$sample)

merged_df <- combined_df %>%
  left_join(sample_sheet, by = "sample")   # keeps all rows from df1

In [ ]:
head(merged_df)

## Find thresholds

In [ ]:
title = "RFP_GFP_scatter"

w <- 5
h <- 5
options(repr.plot.width=w, repr.plot.height=h)

ggscatter = ggplot(merged_df, aes(x = GFP_mean, y = RFP_mean, color = image)) +
  geom_point(alpha = 0.6, size = 0.6) +
  labs(x = "GFP intensity", y = "RFP intensity", title = "") +
  settheme
  #scale_color_manual(values=col_condition)
#ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

## Reformat and process for each sample

In [ ]:
colnames(merged_df)

In [ ]:
df_sample <- merged_df %>%
  group_by(sample, image) %>%
  summarise(
    # carry over metadata from combined_df (pick first non-NA)
    sample_name = dplyr::first(na.omit(sample_name)),
    timepoint   = dplyr::first(na.omit(timepoint)),
    condition   = dplyr::first(na.omit(condition)),
    state       = dplyr::first(na.omit(state)),

    # metrics
    average_vol_um3 = mean(Volume_um3, na.rm = TRUE),
    total_cells     = n(),
    RFPpos          = sum(PosClass == "RFP+", na.rm = TRUE),
    GFPpos          = sum(PosClass == "GFP+", na.rm = TRUE),
    propRFP         = RFPpos / total_cells,
    propGFP         = GFPpos / total_cells,
    .groups = "drop"
  )


In [ ]:
df_sample

## Plot 

In [ ]:
library(ggplot2)
library(dplyr)
library(reshape2)
library(stringr)
library(ggpubr)
library(patchwork)

In [ ]:
# define const for visualization
FONT.SIZE <- 9
LABEL.FONT.SIZE <- 9
w <- 2
h <- 2.5
LINE.W <- 0.3 # equivalent to 0.75pt in Keynote

settheme <- theme_minimal() + theme(
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    #axis.line = element_blank(),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    axis.text.x = element_text(colour = "black", angle = 0,size = LABEL.FONT.SIZE),
    axis.title.y = element_text(margin = margin(r = 3)),
    legend.position="right",
    title = element_text(size = FONT.SIZE) )

In [ ]:
state_cols <- c("Bad" = "#FF9300", "Good" = "#8530FF", "-" = "#424242")

col_condition = c("GFPrev" = "#285F62", 
               "RFPrev" = "#CA4F33", 
               "mosaic"= "#E2A557")

df_sample$state <- factor(df_sample$state, levels = names(state_cols))

### total cell numbers

In [ ]:
plot_bars(subset(df_sample, timepoint == "D6"))

In [ ]:
# total cell numbers
plot_bars <- function(
  df
) {
  p = ggplot(df, aes(x = condition, y = total_cells, group = state)) +  # dots for each file
  stat_summary(
    fun = mean, 
    geom = "bar",
    position = position_dodge(width = 0.75),
    fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
  facet_wrap(~ timepoint, scales = "free_y", nrow = 1) +
  geom_jitter(
    aes(color = state),
    position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
    size = 2, alpha = 0.8
  )  +  # average bar
  stat_summary(
    fun.data = mean_se, 
    geom = "errorbar",
    position = position_dodge(width = 0.75),
    width = 0.2, 
    color = "black")+
  scale_color_manual(name = "state", values = state_cols, limits = names(state_cols), drop = FALSE)+
  scale_y_continuous(limits = c(0, NA), expand = expansion(mult = c(0, 0.06)))+
  labs(
    title = "",
    y = "number of events",
    x = ""
  )+ settheme+
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1)
  ) 
  p
}


title = "cell_number"

w <- 6
h <- 3
options(repr.plot.width=w, repr.plot.height=h)

#T2_plot = plot_bars(subset(df_sample, timepoint == "D4"))
T3_plot = plot_bars(subset(df_sample, timepoint == "D6"))


#p <- (T2_plot | T3_plot) +
  #plot_layout(widths = c(1.5, 2), guides = "collect") &
  #theme(legend.position = "right")

#p  

ggsave(
  filename = file.path(out_dir, paste0(title, ".pdf")),
  plot = p, width = w, height = h
)

In [ ]:
# percentage GFPpositive
plot_bars <- function(
  df
) {
  p = ggplot(df, aes(x = condition, y = propGFP, group = state)) +  # dots for each file
  stat_summary(
    fun = mean, 
    geom = "bar",
    position = position_dodge(width = 0.75),
    fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
  facet_wrap(~ timepoint, scales = "free_y", nrow = 1) +
  geom_jitter(
    aes(color = state),
    position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
    size = 2, alpha = 0.8
  )  +  # average bar
  stat_summary(
    fun.data = mean_se, 
    geom = "errorbar",
    position = position_dodge(width = 0.75),
    width = 0.2, 
    color = "black")+
  scale_color_manual(name = "state", values = state_cols, limits = names(state_cols), drop = FALSE)+
  scale_y_continuous(limits = c(0, NA), expand = expansion(mult = c(0, 0.06)))+
  labs(
    title = "",
    y = "prop GFP+",
    x = ""
  )+ settheme+
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1)
  ) 
  p
}


title = "propGFPpos"

w <- 6
h <- 3
options(repr.plot.width=w, repr.plot.height=h)

T2_plot = plot_bars(subset(df_sample, timepoint == "T2"))
T3_plot = plot_bars(subset(df_sample, timepoint == "T3"))


p <- (T2_plot | T3_plot) +
  plot_layout(widths = c(1.5, 2), guides = "collect") &
  theme(legend.position = "right")

p  
T2_plot
ggsave(
  filename = file.path(out_dir, paste0(title, ".pdf")),
  plot = p, width = w, height = h
)

In [ ]:
# percentage RFPpositive
plot_bars <- function(
  df
) {
  p = ggplot(df, aes(x = condition, y = propGFP, group = state)) +  # dots for each file
  stat_summary(
    fun = mean, 
    geom = "bar",
    position = position_dodge(width = 0.75),
    fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
  facet_wrap(~ timepoint, scales = "free_y", nrow = 1) +
  geom_jitter(
    aes(color = state),
    position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
    size = 2, alpha = 0.8
  )  +  # average bar
  stat_summary(
    fun.data = mean_se, 
    geom = "errorbar",
    position = position_dodge(width = 0.75),
    width = 0.2, 
    color = "black")+
  scale_color_manual(name = "state", values = state_cols, limits = names(state_cols), drop = FALSE)+
  scale_y_continuous(limits = c(0, NA), expand = expansion(mult = c(0, 0.06)))+
  labs(
    title = "",
    y = "prop GFP+",
    x = ""
  )+ settheme+
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1)
  ) 
  p
}


In [ ]:
plot_bars(subset(df_sample, timepoint == "D6"))